This Jupyter notebook contains usual libraries and settings at the beginning of a Python program in Astronomy, for the sake of copy-and-paste.

In [ ]:
# =========================================
# Environment Bootstrap for Astro/Imaging Work
# - Clean, grouped imports
# - Consistent plotting style (LaTeX-like, no LaTeX)
# - Jupyter magics (kept as percent-form lines)
# - Logging + timing helpers
# =========================================

# -------------------------
# Standard Library Imports
# -------------------------
import copy, gc, glob, json, multiprocessing, os, random, re, shutil, sys, time, warnings, logging, numbers, pickle
from importlib import reload
from pprint import pprint

# -------------------------
# Numeric and Plotting
# -------------------------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle

# -------------------------
# Astronomy Stack (Astropy + friends)
# -------------------------
import astropy.constants as ac
import astropy.units as u
from astropy import table
from astropy.convolution import Box1DKernel, convolve
from astropy.coordinates import SkyCoord, concatenate
from astropy.io import fits, ascii
from astropy.modeling import functional_models, models
from astropy.nddata import Cutout2D, NDData
from astropy.stats import sigma_clipped_stats, SigmaClip
from astropy.table import Table
from astropy.wcs import WCS, FITSFixedWarning
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.visualization import astropy_mpl_style, AsinhStretch
warnings.simplefilter('ignore', FITSFixedWarning)

# -------------------------
# Plotting Style (LaTeX-like, no LaTeX)
# -------------------------
import scienceplots
plt.style.use(astropy_mpl_style)
plt.style.use(['bright', 'science', 'no-latex', 'notebook'])
# Font & figure defaults (no external LaTeX)
plt.rcParams.update({
    'text.usetex': False,
    'mathtext.fontset': 'cm',
    'mathtext.default': 'regular',
    'font.family': 'serif',
    # image / figure
    'image.cmap': 'gist_heat',
    'image.origin': 'lower',
    'image.interpolation': 'none',
    'figure.figsize': [6, 6],
    # typography
    'font.size': 12,
    'axes.titlesize': 20,
    'axes.labelsize': 20,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 9,
    'legend.title_fontsize': 12,
})

# -------------------------
# Other Printing and Display Options
# -------------------------
from tqdm import tqdm
np.set_printoptions(precision=3, suppress=True)

# -------------------------
# Logger Setup
# -------------------------
LOG = logging.getLogger(__name__)
LOG.setLevel(logging.INFO)
if not LOG.handlers:
    _console = logging.StreamHandler()
    _fmt = logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s', datefmt='%H:%M:%S')
    _console.setFormatter(_fmt)
    LOG.addHandler(_console)

# -------------------------
# Tracing Helper
# -------------------------
def log_call(func):
    def wrapper(*args, **kwargs):
        LOG.info(f'Calling "{func.__name__}"')
        return func(*args, **kwargs)
    return wrapper
# Usage: @log_call ...

# -------------------------
# Timing Helper (example)
# -------------------------
# t0 = time.perf_counter()
# # ... your code here ...
# elapsed_ms = (time.perf_counter() - t0)*1000
# print(__name__, f'{elapsed_ms:.2f} ms')

# -------------------------
# Jupyter Magics
# -------------------------
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

In [ ]:
# =========================================
# Packages with Less Frequency
# =========================================

# -------------------------
# Third-Party: Imaging and Interpolation
# -------------------------
import cv2
from PIL import Image
from scipy import interpolate

# -------------------------
# Other Astronomy Packages
# -------------------------
from photutils.detection import DAOStarFinder
from photutils.psf import EPSFBuilder, extract_stars
from astroquery.sdss import SDSS

# -------------------------
# Detectron2-related
# -------------------------
import torch
import torchvision
import detectron2
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.engine import DefaultPredictor, DefaultTrainer, SimpleTrainer
from detectron2.modeling import *
from detectron2.solver import *
from detectron2.utils import comm
from detectron2.utils.visualizer import Visualizer
from numpyencoder import NumpyEncoder